In [ ]:
# Day 18: Cross-Frame Player Matching — Nearest Centroid Tracking

In [1]:
!pip install -q ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 36.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 5.0 MB/s eta 0:00:00


In [2]:
import cv2
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image as PILImage
from ultralytics import YOLO
import pandas as pd
model = YOLO('yolov8n.pt')
NaN = np.nan

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [19]:
cap = cv2.VideoCapture("/content/football_clip.mp4")
ret,frame = cap.read()
ret2, frame2 = cap.read()   # grab the next frame after the one already used

In [20]:
def get_centroids(frame):
  result = model.predict(frame)
  boxes = result[0].boxes.xyxy.cpu().numpy()
  centroid = []

  for box in boxes :
    x1,y1,x2,y2 = box
    cx = (x1+x2)/2
    cy = (y1+y2)/2
    centroid.append((cx,cy))
  return centroid

centroid_frame1 = get_centroids(frame)
centroid_frame2 = get_centroids(frame2)




0: 384x640 7 persons, 125.5ms
Speed: 4.6ms preprocess, 125.5ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 7 persons, 123.4ms
Speed: 3.6ms preprocess, 123.4ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)


In [21]:
print(centroid_frame1)
print(centroid_frame2)

[(np.float32(1309.5), np.float32(1181.5613)), (np.float32(2138.3008), np.float32(1163.9364)), (np.float32(3153.3193), np.float32(1180.0847)), (np.float32(2373.5881), np.float32(1165.3231)), (np.float32(1808.2808), np.float32(1148.4467)), (np.float32(1200.4038), np.float32(1154.464)), (np.float32(3742.0144), np.float32(1186.4226))]
[(np.float32(1310.1368), np.float32(1180.262)), (np.float32(2138.9731), np.float32(1163.5256)), (np.float32(3153.0117), np.float32(1179.8584)), (np.float32(2374.056), np.float32(1165.1177)), (np.float32(1808.4924), np.float32(1148.9598)), (np.float32(1198.5328), np.float32(1153.611)), (np.float32(3742.6655), np.float32(1186.3446))]


In [27]:
import math
point = centroid_frame2[0]
matches = []

for point in centroid_frame2:
  best_distance = float('inf')   # nothing can be closer than "infinitely far", so first real d always wins
  best_match = None

  for candidate in centroid_frame1:
    d = math.dist(point,candidate)
    if d<best_distance:
      best_distance = d       # replace with the new smaller distance
      best_match = candidate  # remember which point gave us that distance
  matches.append((point,best_match))

print(matches)


[((np.float32(1310.1368), np.float32(1180.262)), (np.float32(1309.5), np.float32(1181.5613))), ((np.float32(2138.9731), np.float32(1163.5256)), (np.float32(2138.3008), np.float32(1163.9364))), ((np.float32(3153.0117), np.float32(1179.8584)), (np.float32(3153.3193), np.float32(1180.0847))), ((np.float32(2374.056), np.float32(1165.1177)), (np.float32(2373.5881), np.float32(1165.3231))), ((np.float32(1808.4924), np.float32(1148.9598)), (np.float32(1808.2808), np.float32(1148.4467))), ((np.float32(1198.5328), np.float32(1153.611)), (np.float32(1200.4038), np.float32(1154.464))), ((np.float32(3742.6655), np.float32(1186.3446)), (np.float32(3742.0144), np.float32(1186.4226)))]


In [28]:
print(matches)

[((np.float32(1310.1368), np.float32(1180.262)), (np.float32(1309.5), np.float32(1181.5613))), ((np.float32(2138.9731), np.float32(1163.5256)), (np.float32(2138.3008), np.float32(1163.9364))), ((np.float32(3153.0117), np.float32(1179.8584)), (np.float32(3153.3193), np.float32(1180.0847))), ((np.float32(2374.056), np.float32(1165.1177)), (np.float32(2373.5881), np.float32(1165.3231))), ((np.float32(1808.4924), np.float32(1148.9598)), (np.float32(1808.2808), np.float32(1148.4467))), ((np.float32(1198.5328), np.float32(1153.611)), (np.float32(1200.4038), np.float32(1154.464))), ((np.float32(3742.6655), np.float32(1186.3446)), (np.float32(3742.0144), np.float32(1186.4226)))]


In [ ]:
# **Day 18: Cross-Frame Player Matching — Nearest Centroid Tracking**

# - Detected all players in a frame using YOLO, filtered for the "person" class
# - Reduced each player's bounding box to a single centroid point
# - Wrapped detection + centroid logic into a reusable `get_centroids(frame)` function
# - Grabbed a second frame and computed its centroids too
# - For each player in frame 2, found the closest-distance centroid in frame 1 using `math.dist`
# - Produced a matched list linking each frame-2 player to their frame-1 position
# - This nearest-neighbor matching is the foundation for real tracking —
#    ID assignment, speed, and possession all build on it next